# Phase 3: Scam Text Classifier

In [1]:
# Imports
import html
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [2]:
# Load and clean the data (same cleaning as the Phase 2 notebook)
def load_data(path):
    df = pd.read_csv(path)
    df["LABEL"] = df["LABEL"].str.strip().str.lower()
    df["TEXT"] = df["TEXT"].str.replace("&lt;#&gt;", " ", regex=False).str.strip().apply(html.unescape)
    for c in ["URL", "EMAIL", "PHONE"]:
        df[c] = df[c].str.lower().eq("yes")
    return df

df = load_data("../data/Dataset_5971.csv")
print("Raw rows:", len(df))

df = df.drop_duplicates().reset_index(drop=True)
print("After removing duplicates:", len(df))

Raw rows: 5971
After removing duplicates: 5876


In [3]:
# Target: ham = 0, scam (spam + smishing) = 1
df["scam"] = (df["LABEL"] != "ham").astype(int)

counts = df["scam"].value_counts()
print(counts)
print("Share of scams:", round(counts[1] / len(df) * 100, 1), "%")

scam
0    4834
1    1042
Name: count, dtype: int64
Share of scams: 17.7 %


In [4]:
# Split into training (80%) and test (20%) sets
# stratify keeps the same ham/scam mix in both sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["scam"])

print("Training messages:", len(train_df), "| scam share:", round(train_df["scam"].mean() * 100, 1), "%")
print("Test messages:", len(test_df), "| scam share:", round(test_df["scam"].mean() * 100, 1), "%")

Training messages: 4700 | scam share: 17.7 %
Test messages: 1176 | scam share: 17.8 %


Creating a testing out different models:

In [5]:
# Model A: TF-IDF words + Naive Bayes (baseline)
tfidf = TfidfVectorizer(lowercase=True, stop_words="english", min_df=2, max_df=0.9)
X_train_words = tfidf.fit_transform(train_df["TEXT"])   # learn vocabulary from training only
X_test_words  = tfidf.transform(test_df["TEXT"])        # same vocabulary for test
y_train, y_test = train_df["scam"], test_df["scam"]

print("Number of word features:", X_train_words.shape[1])

nb = MultinomialNB()
nb.fit(X_train_words, y_train)
nb_pred = nb.predict(X_test_words)

print(classification_report(y_test, nb_pred, target_names=["ham", "scam"], digits=3))

Number of word features: 3557
              precision    recall  f1-score   support

         ham      0.973     0.998     0.985       967
        scam      0.989     0.871     0.926       209

    accuracy                          0.975      1176
   macro avg      0.981     0.934     0.956      1176
weighted avg      0.976     0.975     0.975      1176



In [7]:
# Model B (part 1): Phase 2 text signals as features
# Phase 2 text signals as features (one row per message)
sia = SentimentIntensityAnalyzer()

def signal_features(texts):
    t = pd.Series(texts).reset_index(drop=True)
    return pd.DataFrame({
        "words":          t.str.split().str.len(),
        "digits":         t.str.count(r"\d"),
        "has_phone":      t.str.contains(r"\d{10,}").astype(int),
        "has_shortcode":  t.str.contains(r"\b\d{4,6}\b").astype(int),
        "has_url":        t.str.contains(r"http|www\.|\.com|\.co\.uk|\.ly|\.io", case=False).astype(int),
        "has_currency":   t.str.contains(r"[£$€]").astype(int),
        "caps_pct":       t.str.count(r"[A-Z]") / t.str.len().clip(lower=1) * 100,
        "exclaim":        t.str.count("!"),
        "call_to_action": t.str.contains(r"\b(?:call|txt|text|reply|click|send|visit)\b", case=False).astype(int),
        "sentiment":      t.apply(lambda x: sia.polarity_scores(x)["compound"]),
    })

S_train = signal_features(train_df["TEXT"])
S_test  = signal_features(test_df["TEXT"])
S_train.groupby(y_train.values).mean().round(2)

,words,digits,has_phone,has_shortcode,has_url,has_currency,caps_pct,exclaim,call_to_action,sentiment
0,14.04,0.32,0.00,0.00,0.00,0.00,5.73,0.18,0.10,0.17
1,23.00,15.05,0.58,0.44,0.22,0.27,11.00,0.59,0.77,0.45


In [8]:
# Model B (part 2): scale signals, combine with TF-IDF words, train logistic regression
from sklearn.preprocessing import StandardScaler

# Put the signals on the same scale (fit on training only)
scaler = StandardScaler()
S_train_scaled = scaler.fit_transform(S_train)
S_test_scaled  = scaler.transform(S_test)

# Words + signals side by side
X_train_B = hstack([X_train_words, csr_matrix(S_train_scaled)]).tocsr()
X_test_B  = hstack([X_test_words,  csr_matrix(S_test_scaled)]).tocsr()

# Model B: logistic regression
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_B, y_train)
lr_pred = logreg.predict(X_test_B)

print(classification_report(y_test, lr_pred, target_names=["ham", "scam"], digits=3))

              precision    recall  f1-score   support

         ham      0.979     0.996     0.987       967
        scam      0.979     0.900     0.938       209

    accuracy                          0.979      1176
   macro avg      0.979     0.948     0.962      1176
weighted avg      0.979     0.979     0.978      1176

